In [ ]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm # For multimodel support
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

In [ ]:
import os 
from dotenv import load_dotenv
import requests
import pandas as pd

load_dotenv()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

def webscrape(product):
 
    url = "https://api.tavily.com/search"

    payload = {
        "query": product ,
        "topic": "general",
        "search_depth": "advanced",
        "chunks_per_source": 3,
        "max_results": 100,
        "time_range": None,
        "days": 3,
        "include_answer": True,
        "include_raw_content": False,
        "include_images": False,
        "include_image_descriptions": False,
        # "include_domains": ["https://www.reddit.com/", "https://www.pcgamer.com/","https://www.ign.com/news","https://www.reddit.com/r/GamingChairReviews","https://www.techgearlab.com","https://www.tomshardware.com","https://chairdeskexpert.com","https://www.buzzfeed.com" ],
        "include_domains": [],
        "exclude_domains": []
    }
    headers = {
        "Authorization": "Bearer " + TAVILY_API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.request("POST", url, json=payload, headers=headers)

    response = response.json()
    print(response)
    answer = response["answer"]
    results_dict = {}
    for result in response["results"]:
        results_dict[result["title"]] = {"url":result["url"], 
                                         "content":result["content"]}
    return answer, results_dict






# product = "Secretlab Titan Evo Lite gaming chair review"
# product = "gaming chair positive product "
# product = "Scrape online reviews for top gaming chairs. Extract common complaints on comfort, durability, and ergonomics. Focus on armrests, lumbar support, and materials."

# product ="Compare gaming chairs vs. ergonomic office chairs  on adjustability, breathability, and posture support. Highlight gaps in gaming chair designs."
# product ="Find technical specs for gaming chair materials (examples: PU leather, memory foam density, frame alloys). Include stress-test results or warranty claims."
# product ="Scrape user complaints about gaming chair assembly (tools, instructions, part alignment)"
product ="gaming chair design issues"

webscrape_result = webscrape(product)
print(webscrape_result)

In [ ]:
AGENT_MODEL = "gemini-1.5-pro-latest" # Starting with a powerful Gemini model
web_scrapper_agent = Agent(
name="web_scrapper_v1",
model=AGENT_MODEL, # Specifies the underlying LLM
description="Provides weather information for specific cities.", # Crucial for delegation later
instruction="You are a prompt web scrapper agenet. Your primary goal is to create prompt that send to tavily api that is able data about the product in regards to product design review, how to improve the products, functional design specification, and competitors. "
"you MUST use the 'webscrape' tool to find the information. "
"Analyze the tool's response: if the status is 'error', inform the user politely about the error message. "
"If the status is 'success', present the weather 'report' clearly and concisely to the user. ",
tools=[webscrape], # Make the tool available to this agent
)
print(f"Agent '{web_scrapper_agent.name}' created using model'{AGENT_MODEL}'.")

In [ ]:
session_service = InMemorySessionService()
# Define constants for identifying the interaction context
APP_NAME = "mehmeh"
USER_ID = "user_1"
SESSION_ID = "session_001" # Using a fixed ID for simplicity
# Create the specific session where the conversation will happen
session = session_service.create_session(
app_name=APP_NAME,
user_id=USER_ID,
session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")
# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(
agent=web_scrapper_agent, # The agent we want to run
app_name=APP_NAME, # Associates runs with our app
session_service=session_service # Uses our session
manager
)
print(f"Runner created for agent '{runner.agent.name}'.")


In [ ]:
import asyncio
from google.genai import types # For creating message
Content/Parts
async def call_agent_async(query: str):
"""Sends a query to the agent and prints the final
response."""
print(f"\n>>> User Query: {query}")
# Prepare the user's message in ADK format
content = types.Content(role='user', parts=
[types.Part(text=query)])
final_response_text = "Agent did not produce a final
response." # Default
# Key Concept: run_async executes the agent logic and
yields Events.
# We iterate through events to find the final answer.
async for event in runner.run_async(user_id=USER_ID,
session_id=SESSION_ID, new_message=content):
# You can uncomment the line below to see *all* events
during execution
# print(f" [Event] Author: {event.author}, Type:
{type(event).__name__}, Final: {event.is_final_response()},
Content: {event.content}")
# Key Concept: is_final_response() marks the concluding
message for the turn.
if event.is_final_response():
if event.content and event.content.parts:
# Assuming text response in the first part
final_response_text =
event.content.parts[0].text
elif event.actions and event.actions.escalate: #
Handle potential errors/escalations
final_response_text = f"Agent escalated:
{event.error_message or 'No specific message.'}"
# Add more checks here if needed (e.g., specific
error codes)
break # Stop processing events once the final
response is found
print(f"<<< Agent Response: {final_response_text}"

In [ ]:
# pip install google-adk

In [ ]:
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.agents.llm_agent import LlmAgent
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

# --- Constants ---
APP_NAME = "code_pipeline_app"
USER_ID = "dev_user_01"
SESSION_ID = "pipeline_session_01"
GEMINI_MODEL = "gemini-2.0-flash"

# --- 1. Define Sub-Agents for Each Pipeline Stage ---

# Code Writer Agent
# Takes the initial specification (from user query) and writes code.
code_writer_agent = LlmAgent(
    name="CodeWriterAgent",
    model=GEMINI_MODEL,
    instruction="""You are a Code Writer AI.
    Based on the user's request, write the initial Python code.
    Output *only* the raw code block.
    """,
    description="Writes initial code based on a specification.",
    # Stores its output (the generated code) into the session state
    # under the key 'generated_code'.
    output_key="generated_code"
)

# Code Reviewer Agent
# Takes the code generated by the previous agent (read from state) and provides feedback.
code_reviewer_agent = LlmAgent(
    name="CodeReviewerAgent",
    model=GEMINI_MODEL,
    instruction="""You are a Code Reviewer AI.
    Review the Python code provided in the session state under the key 'generated_code'.
    Provide constructive feedback on potential errors, style issues, or improvements.
    Focus on clarity and correctness.
    Output only the review comments.
    """,
    description="Reviews code and provides feedback.",
    # Stores its output (the review comments) into the session state
    # under the key 'review_comments'.
    output_key="review_comments"
)

# Code Refactorer Agent
# Takes the original code and the review comments (read from state) and refactors the code.
code_refactorer_agent = LlmAgent(
    name="CodeRefactorerAgent",
    model=GEMINI_MODEL,
    instruction="""You are a Code Refactorer AI.
    Take the original Python code provided in the session state key 'generated_code'
    and the review comments found in the session state key 'review_comments'.
    Refactor the original code to address the feedback and improve its quality.
    Output *only* the final, refactored code block.
    """,
    description="Refactors code based on review comments.",
    # Stores its output (the refactored code) into the session state
    # under the key 'refactored_code'.
    output_key="refactored_code"
)

# --- 2. Create the SequentialAgent ---
# This agent orchestrates the pipeline by running the sub_agents in order.
code_pipeline_agent = SequentialAgent(
    name="CodePipelineAgent",
    sub_agents=[code_writer_agent, code_reviewer_agent, code_refactorer_agent]
    # The agents will run in the order provided: Writer -> Reviewer -> Refactorer
)

# Session and Runner
session_service = InMemorySessionService()
session = session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
runner = Runner(agent=code_pipeline_agent, app_name=APP_NAME, session_service=session_service)


# Agent Interaction
def call_agent(query):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    events = runner.run(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    for event in events:
        if event.is_final_response():
            final_response = event.content.parts[0].text
            print("Agent Response: ", final_response)

call_agent("perform math addition")

In [ ]:
# !pip install google-adk